# Turning Up the Guidance Until It Breaks

**Topic:** Classifier-free guidance &nbsp;·&nbsp; **Difficulty:** Medium

Classifier-free guidance extrapolates away from the unconditional prediction:
`eps = eps_uncond + w * (eps_cond - eps_uncond)`. At `w = 1` you get the
conditional model, and above that you get something sharper than either, which is
why everyone runs `w = 7`.

There is no free lunch. Guidance shrinks the sampled distribution (better prompt
adherence, less diversity), and past a point it distorts it. Measure both effects
on a Gaussian toy where the true answer is known.

## The data (created in the next cell)

- The conditional score corresponds to `N(2.0, 1.0)` and the unconditional to
  `N(0.0, 2.0)`. Both are known exactly. In this toy the unconditional model is
  the broader one, so the guided precision rises with `w` and the breakdown
  described in the title does not appear, so `w_max_valid` is then just the last `w`
  in the list.
- `ws = [0, 1, 2, 4, 7, 12]`.
- 20,000 samples per setting, drawn in the order `ws` lists them from a
  single `np.random.default_rng(88)` reused across settings.

## Your task

With **NumPy only**:

1. For a Gaussian, the guided distribution is again Gaussian. Its precision is
   `w * prec_cond + (1 - w) * prec_uncond` and its mean solves
   `prec_guided * mu_guided = w * prec_cond * mu_cond + (1 - w) * prec_uncond *
   mu_uncond`. Compute these in closed form for each `w`.
2. Sample 20,000 draws from each guided distribution.
3. Record, for each `w`: `mean`, `std`, and `frac_on_target`, the fraction of
   samples within 1.0 of the conditional mean `2.0`.
4. Set `answers` to a dict with `"mu_guided"`, `"sigma_guided"`,
   `"frac_on_target"` (6 values each), and `"w_max_valid"`, the largest `w` in
   the list whose guided precision is still positive.

## Evaluation

Compared to the reference to a relative tolerance of 1e-5. The evaluation cell
checks adherence rises and diversity falls as `w` grows.

In [ ]:
import numpy as np

mu_c, sd_c = 2.0, 1.0
mu_u, sd_u = 0.0, 2.0
ws = [0, 1, 2, 4, 7, 12]
n = 20000
print(f"conditional N({mu_c}, {sd_c}^2), unconditional N({mu_u}, {sd_u}^2)")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("mu_guided", [6], 11.887301887301888, 6.276701126719228, 1e-05),
    ("sigma_guided", [6], 5.0658285495656195, 0.4106786967598909, 1e-05),
    ("frac_on_target", [6], 4.28165, 2.0513443653183203, 1e-05),
    ("w_max_valid", [], 12.0, -2.00925617777399, 1e-05),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "sigma_guided" in answers:
    _sd = np.asarray(answers["sigma_guided"], dtype=float)
    _fr = np.asarray(answers["frac_on_target"], dtype=float)
    _tight = _sd[4] < _sd[1]
    _adhere = _fr[4] > _fr[1]
    print(f"  {'guidance tightens':<24} {'ok' if _tight else 'wrong'}  "
          f"(sigma {_sd[1]:.3f} at w=1 -> {_sd[4]:.3f} at w=7)")
    print(f"  {'adherence rises':<24} {'ok' if _adhere else 'wrong'}  "
          f"({_fr[1]:.3f} -> {_fr[4]:.3f})")
    _ok = _ok and _tight and _adhere

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Guidance is a linear combination of scores, and for Gaussians a linear
combination of scores is again a Gaussian score with precision
`w * prec_c + (1-w) * prec_u`. That formula makes the trade explicit: raising `w`
raises the precision, so the standard deviation falls and samples pile up near the
conditional mean: better adherence, less diversity. It also shows the failure
mode: if `prec_u > prec_c`, a large enough `w` drives the precision negative and
the "distribution" stops existing, which is the toy-model version of the
saturated, over-contrasted images high guidance produces.

```python
import numpy as np

prec_c = 1.0 / sd_c ** 2
prec_u = 1.0 / sd_u ** 2

rng = np.random.default_rng(88)

mus, sds, fracs = [], [], []
valid = []
for w in ws:
    prec = w * prec_c + (1 - w) * prec_u
    if prec <= 0:
        mus.append(np.nan); sds.append(np.nan); fracs.append(np.nan)
        continue
    valid.append(w)
    mu = (w * prec_c * mu_c + (1 - w) * prec_u * mu_u) / prec
    sd = float(np.sqrt(1.0 / prec))
    s = rng.normal(mu, sd, n)
    mus.append(float(mu))
    sds.append(sd)
    fracs.append(float((np.abs(s - mu_c) < 1.0).mean()))

answers = {
    "mu_guided": np.array(mus),
    "sigma_guided": np.array(sds),
    "frac_on_target": np.array(fracs),
    "w_max_valid": float(max(valid)),
}
```